In [1]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder
import wandb
from sklearn.metrics import f1_score
from tqdm import tqdm

In [2]:
# .npz laden
train = np.load("../Model_data/train.npz")
test  = np.load("../Model_data/test.npz")

X_train, y_train = train["X"], train["y"]
X_test,  y_test  = test["X"],  test["y"]

# X_train hat die Form (Anzahl Windows, Zeitschritte pro Window, Anzahl Features)
print(X_train.shape)
print(y_train.shape)

(16782, 100, 13)
(16782,)


In [3]:
# PyTorch braucht numerische Klassen
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

# NumPy zu PyTorch Tensoren und umformen zu (N, 13, 100) (Chanels in der Mitte) für CNN, nicht bei LSTM
X_train_t = torch.tensor(X_train, dtype=torch.float32).permute(0, 2, 1)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32).permute(0, 2, 1)

y_train_t = torch.tensor(y_train_enc, dtype=torch.long)
y_test_t  = torch.tensor(y_test_enc,  dtype=torch.long)

# Datasets erstellen als Tensoren
train_ds = TensorDataset(X_train_t, y_train_t)
test_ds  = TensorDataset(X_test_t,  y_test_t)

# GPU falls verfügbar
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [4]:
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\jessi\_netrc.
wandb: Currently logged in as: jessischmid01 (CDL1_team) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# 1D-CNN Aufbau

In der Version 2 "1D-CNN_v2..." wurde die Data Augmentation und ein Learningrate scheduler eingebaut, um das Training stabiler zu machen. 

In [5]:
class CNN1D(nn.Module):
    def __init__(self, n_features, n_classes, dropout=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(n_features, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        return self.classifier(self.net(x))

In [6]:
def evaluate(model, loader, criterion, device):
    """
    Gibt loss, accuracy und f1-macro für einen DataLoader zurück.
    Wird für Train- und Validation-Set verwendet.
    """
    model.eval()
    total_loss, correct = 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            preds = model(X_batch)
            total_loss += criterion(preds, y_batch).item()
            correct += (preds.argmax(1) == y_batch).sum().item()
            all_preds.extend(preds.argmax(1).cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    n = len(loader.dataset)
    avg_loss = total_loss / len(loader)
    acc = correct / n
    f1 = f1_score(all_labels, all_preds, average="macro")

    return {"loss": avg_loss, "acc": acc, "f1": f1}

def augment(X_batch):
    """Gaussian Noise + Amplitude Scaling direkt auf dem Batch-Tensor."""
    # Gaussian Noise
    X_batch = X_batch + torch.randn_like(X_batch) * 0.01
    # Amplitude Scaling: pro Sample ein zufälliger Faktor zwischen 0.9 und 1.1
    scale = 0.9 + torch.rand(X_batch.size(0), 1, 1, device=X_batch.device) * 0.2
    X_batch = X_batch * scale
    return X_batch

def train(config, train_ds, test_ds, device, project="CDL1"):
    """
    Trainiert ein CNN1D Modell mit den gegebenen Hyperparametern.

    Args:
        config: Dictionary mit allen Hyperparametern
        train_loader: DataLoader für Trainingsdaten
        test_loader: DataLoader für Validierungsdaten
        device: torch.device (cuda / cpu)
        project: W&B Projektname

    Returns:
        model: Trainiertes Modell (bestes nach val/f1)
        history: Dictionary mit Metriken pro Epoch
    """
    # Wandb Run initialisieren
    fold_suffix = f"_Fold{config['fold']}" if "fold" in config else ""
    run = wandb.init(
        project=project,
        entity="CDL1_team",
        config=config,
        name=f"1D-CNN_v2_lr{config['lr']}_bs{config['batch_size']}_do{config['dropout']}{fold_suffix}"
    )
    cfg = wandb.config

    # DataLoader wird jetzt mit der config Batch Size erstellt
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True)
    test_loader = DataLoader(test_ds,  batch_size=cfg.batch_size, shuffle=False)

    # Modell, Optimizer, Loss festlegen, Adam passt die Lernrate während dem Training an
    model = CNN1D(cfg.n_features, cfg.n_classes, dropout=cfg.dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)

    wandb.watch(model, log="all", log_freq=50)

    # Variablen um beste F1 Scores und die dazugehörigen Gewichte zu speichern
    history = {"train": [], "val": []}
    best_f1 = 0.0
    best_state = None

    # hinzufügen von tqdm, damit man den Fortschritt sieht
    epoch_bar = tqdm(range(cfg.epochs), desc="Training", unit="epoch")

    for epoch in epoch_bar:

        model.train()
        train_loss, train_correct = 0, 0
        all_preds, all_labels = [], []

        batch_bar = tqdm(train_loader, desc=f"  Epoch {epoch+1:02d}", leave=False, unit="batch")

        for X_batch, y_batch in batch_bar:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            X_batch = augment(X_batch)

            optimizer.zero_grad()
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_correct += (preds.argmax(1) == y_batch).sum().item()
            all_preds.extend(preds.argmax(1).cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
            
            # Aktuellen Loss direkt im Batch-Balken anzeigen
            batch_bar.set_postfix(loss=f"{loss.item():.3f}")

        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]

        train_metrics = {
            "loss": train_loss / len(train_loader),
            "acc": train_correct / len(train_loader.dataset),
            "f1": f1_score(all_labels, all_preds, average="macro"),
        }

        # Validation
        val_metrics = evaluate(model, test_loader, criterion, device)

        # Bestes Modell speichern
        if val_metrics["f1"] > best_f1:
            best_f1 = val_metrics["f1"]
            best_state = model.state_dict().copy()

        # History & W&B loggen 
        history["train"].append(train_metrics)
        history["val"].append(val_metrics)

        wandb.log({
            "epoch": epoch + 1,
            "train/loss": train_metrics["loss"],
            "train/acc": train_metrics["acc"],
            "train/f1": train_metrics["f1"],
            "val/loss": val_metrics["loss"],
            "val/acc": val_metrics["acc"],
            "val/f1": val_metrics["f1"],
            "train/lr": current_lr,
        })

        # Epoch-Balken mit finalen Metriken aktualisieren
        epoch_bar.set_postfix({
            "train_f1": f"{train_metrics['f1']:.3f}",
            "val_f1": f"{val_metrics['f1']:.3f}",
            "val_acc": f"{val_metrics['acc']:.3f}",
        })

    # Bestes Modell laden und zurückgeben
    model.load_state_dict(best_state)
    wandb.finish()

    return model, history

In [7]:
# windowing_size und step_size sind im sliding window der Datenbearbeitung vorhanden und dienen nur der Dokumentation
config = {
    "model": "1D-CNN",
    "window_size": 100,
    "step_size": 50,
    "batch_size": 64,
    "lr": 0.001,
    "epochs": 30,
    "n_features": 13,
    "n_classes": len(le.classes_),
    "dropout": 0.4
}

model, history = train(
    config = config,
    train_ds = train_ds,
    test_ds = test_ds,
    device = device,
    project = "CDL1"
)

Training: 100%|██████████| 30/30 [01:27<00:00,  2.91s/epoch, train_f1=0.980, val_f1=0.951, val_acc=0.969]


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/acc,▁▅▆▆▆▆▆▇▇▆▇▇▇▇▇▇▇▇▇███████████
train/f1,▁▅▆▆▆▆▆▇▇▆▇▇▇▇▇▇▇▇▇███████████
train/loss,█▄▄▃▃▃▃▂▂▃▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁
train/lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
val/acc,▁▂▂▂▄▄▃▃▃▅▄▄▅▇▅▇▇▇▇▇▇▇▇▇██████
val/f1,▁▂▃▁▄▄▃▄▄▅▄▅▅▆▅▇▇▇▇▇▇▇▇▇█▇████
val/loss,▇▇█▇▆▃▆▆▅▁▆▅▅▃▆▅▅▃▅▆▄▄▃▃▃▃▃▃▃▃
epoch,30
train/acc,0.98689
train/f1,0.97984


# Erwartungen Gridsearch

Batchsize beeinflusst die Varianz der Gradienten, somit sind die Updates ungenauer, wenn wir eine kleine Batchsize haben. Dafür kann man mit tiefen Batchsizes eher einem lokalen Minima entweichen und besser generalisieren. Um Stabilität im Training zu haben, muss dafür bei kleiner Batchsize die Lernrate auch klein sein. Für grössere Batchsizes, können dafür grössere Lernraten verwendet werden. 

Dropout ist eine Regularisierungsmethode, bei dem im Fully connected Layer eine gewisse Prozentzahl an Neuronen zufällig ausgeschalten werden. So sollte das Overfitting reduziert werden. 

Folglich sollten folgende Kombinationen gut funktionieren:

lr: 0.01 + bs: 128 + do: 0.4
lr: 0.001 + bs: 64 + do: 0.4
lr: 0.0001 + bs: 32 + do: 0.4

In [8]:
import itertools

# Grid 1 definieren
param_grid = {
    "lr":         [0.01, 0.001, 0.0001],
    "batch_size": [32, 64, 128],
    "dropout":    [0.2, 0.4, 0.6],
}

# Alle Kombinationen erzeugen
keys, values = zip(*param_grid.items())
combinations = list(itertools.product(*values))
print(f"Gesamt: {len(combinations)} Kombinationen")

# Ergebnisse sammeln
results = []

for i, combo in enumerate(combinations, 1):
    params = dict(zip(keys, combo))
    print(f"\n[{i}/{len(combinations)}] {params}")

    cfg = {
        "model":       "1D-CNN",
        "window_size": 100,
        "step_size":   50,
        "epochs":      20,
        "n_features":  13,
        "n_classes":   len(le.classes_),
        **params,  # lr, batch_size, dropout werden überschrieben
    }

    model, history = train(
        config   = cfg,
        train_ds = train_ds,
        test_ds  = test_ds,
        device   = device,
        project  = "CDL1",
    )

    # Bestes val/f1 aus der History holen
    best_val_f1  = max(e["f1"]  for e in history["val"])
    best_val_acc = max(e["acc"] for e in history["val"])

    results.append({
        **params,
        "best_val_f1":  best_val_f1,
        "best_val_acc": best_val_acc,
        "model":        model,
        "history":      history,
    })

    print(f"  → best val/f1: {best_val_f1:.4f}  |  best val/acc: {best_val_acc:.4f}")

# Nach F1 sortieren und Top-3 ausgeben
results.sort(key=lambda x: x["best_val_f1"], reverse=True)


Gesamt: 27 Kombinationen

[1/27] {'lr': 0.01, 'batch_size': 32, 'dropout': 0.2}


Training: 100%|██████████| 20/20 [01:48<00:00,  5.41s/epoch, train_f1=0.952, val_f1=0.932, val_acc=0.961]


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train/acc,▁▅▆▆▆▇▇▇▇▇▇▇████████
train/f1,▁▅▆▆▇▇▇▇▇▇▇▇████████
train/loss,█▅▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁
train/lr,███▇▇▇▆▆▅▅▄▃▃▂▂▂▁▁▁▁
val/acc,▁▄▅▅▆▆▆▆▅▇▇▇▇▇█▇▇███
val/f1,▁▁▄▄▅▆▅▅▄▆▆▆▇▇▇▇▇▇██
val/loss,█▇▄▄▄▃▃▃▄▄▂▂▂▂▁▁▁▁▁▁
epoch,20
train/acc,0.96931
train/f1,0.95225


  → best val/f1: 0.9321  |  best val/acc: 0.9608

[2/27] {'lr': 0.01, 'batch_size': 32, 'dropout': 0.4}


Training: 100%|██████████| 20/20 [01:41<00:00,  5.09s/epoch, train_f1=0.940, val_f1=0.902, val_acc=0.948]


...

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train/acc,▁▄▆▇▇▇▇▇████████████
train/f1,▁▄▅▆▇▇▇▇████████████
train/loss,█▄▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
train/lr,███▇▇▇▆▆▅▅▄▃▃▂▂▂▁▁▁▁
val/acc,▁▄▅▆▇▇▇▇█▇██████████
val/f1,▁▄▄▆▇▇▇▇████████████
val/loss,█▅▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
epoch,20
train/acc,0.95704
train/f1,0.92876


  → best val/f1: 0.8993  |  best val/acc: 0.9390

[27/27] {'lr': 0.0001, 'batch_size': 128, 'dropout': 0.6}


Training: 100%|██████████| 20/20 [00:32<00:00,  1.61s/epoch, train_f1=0.926, val_f1=0.901, val_acc=0.940]


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train/acc,▁▄▆▇▇▇▇█████████████
train/f1,▁▄▅▆▇▇▇▇████████████
train/loss,█▄▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
train/lr,███▇▇▇▆▆▅▅▄▃▃▂▂▂▁▁▁▁
val/acc,▁▃▅▆▇▇▇█████████████
val/f1,▁▄▄▆▇▇▇▇█▇██████████
val/loss,█▅▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
epoch,20
train/acc,0.95495
train/f1,0.92576


  → best val/f1: 0.9017  |  best val/acc: 0.9404


In [9]:
print("\n" + "="*60)
print("TOP 3 KONFIGURATIONEN")
print("="*60)
for rank, r in enumerate(results[:3], 1):
    print(f"\n#{rank}  val/f1={r['best_val_f1']:.4f}  acc={r['best_val_acc']:.4f}")
    print(f"     lr={r['lr']}  batch_size={r['batch_size']}  dropout={r['dropout']}")

# Bestes Modell direkt verfügbar
best = results[0]
best_model = best["model"]
print(f"\nBestes Modell geladen: lr={best['lr']}, bs={best['batch_size']}, do={best['dropout']}")


TOP 3 KONFIGURATIONEN

#1  val/f1=0.9525  acc=0.9686
     lr=0.001  batch_size=64  dropout=0.2

#2  val/f1=0.9497  acc=0.9660
     lr=0.001  batch_size=128  dropout=0.4

#3  val/f1=0.9480  acc=0.9701
     lr=0.001  batch_size=32  dropout=0.4

Bestes Modell geladen: lr=0.001, bs=64, do=0.2


# Ergebnise Gridsearch

Das beste Model ist lr = 0.001, batchsize = 64 und dropout = 0.2 mit einem Validation F1-Score von 0.953. Was schon gut ist. 

# Cross-Validation

Als nächstes wird die Cross-Validation 5-Fold auf das Model mit der besten Parameterkombination angewendet. So sieht man auch wie gross die Varianz zwischen den Vorhersagen ist. 

In [10]:
from sklearn.model_selection import StratifiedKFold

best_config = {
    "model":       "1D-CNN",
    "window_size": 100,
    "step_size":   50,
    "epochs":      30,
    "n_features":  13,
    "n_classes":   len(le.classes_),
    "lr":          0.001,
    "batch_size":  64,
    "dropout":     0.2,
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_t, y_train_t), 1):
    print(f"\n{'='*40}\nFold {fold}/5\n{'='*40}")

    fold_train_ds = TensorDataset(X_train_t[train_idx], y_train_t[train_idx])
    fold_val_ds   = TensorDataset(X_train_t[val_idx],   y_train_t[val_idx])

    _, history = train(
        config   = {**best_config, "fold": fold},
        train_ds = fold_train_ds,
        test_ds  = fold_val_ds,
        device   = device,
        project  = "CDL1",
    )

    best_f1  = max(e["f1"]  for e in history["val"])
    best_acc = max(e["acc"] for e in history["val"])
    fold_results.append({"fold": fold, "f1": best_f1, "acc": best_acc})
    print(f"Fold {fold} → F1: {best_f1:.4f} | Acc: {best_acc:.4f}")

print(f"\n{'='*40}\nFinales Modell (alle Trainingsdaten)\n{'='*40}")

final_model, final_history = train(
    config   = {**best_config, "fold": "final"},
    train_ds = TensorDataset(X_train_t, y_train_t),
    test_ds  = TensorDataset(X_test_t,  y_test_t),
    device   = device,
    project  = "CDL1",
)

final_f1  = max(e["f1"]  for e in final_history["val"])
final_acc = max(e["acc"] for e in final_history["val"])

f1s  = [r["f1"]  for r in fold_results]
accs = [r["acc"] for r in fold_results]

print("\n" + "="*40)
print("ERGEBNIS")
print("="*40)
for r in fold_results:
    print(f"  Fold {r['fold']}: F1={r['f1']:.4f} | Acc={r['acc']:.4f}")
print(f"{'─'*40}")
print(f"  CV Mean:  F1={np.mean(f1s):.4f} | Acc={np.mean(accs):.4f}")
print(f"  CV Std:   F1={np.std(f1s):.4f}  | Acc={np.std(accs):.4f}")
print(f"{'─'*40}")
print(f"  Final:    F1={final_f1:.4f} | Acc={final_acc:.4f}  ← Testset")


Fold 1/5


Training: 100%|██████████| 30/30 [01:17<00:00,  2.57s/epoch, train_f1=0.972, val_f1=0.970, val_acc=0.983]


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/acc,▁▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇█████████████
train/f1,▁▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇██████████████
train/loss,█▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
val/acc,▄▆▆▆▆▅▆▇▁▇▇▇▇▇▇▇▇██▇██████████
val/f1,▃▅▆▆▆▅▆▇▁▆▇▇▇▇▆▇▇▇█▇██████████
val/loss,▅▄▃▃▃▃▃▂█▂▂▂▂▂▂▂▂▁▂▂▁▁▁▁▁▁▁▁▁▁
epoch,30
train/acc,0.98197
train/f1,0.97245


Fold 1 → F1: 0.9735 | Acc: 0.9839

Fold 2/5


Training: 100%|██████████| 30/30 [01:10<00:00,  2.36s/epoch, train_f1=0.973, val_f1=0.983, val_acc=0.990]


...

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/acc,▁▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇██████████
train/f1,▁▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇████████████
train/loss,█▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
val/acc,▁▃▃▅▅▅▄▅▅▆▆▆▇▆▆▆▇█▇▇██▇▇██████
val/f1,▁▃▃▄▅▅▂▅▅▆▆▆▇▅▇▆▇██▇█▇▇███████
val/loss,█▅▅▄▃▃▄▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
epoch,30
train/acc,0.98533
train/f1,0.97733


Fold 5 → F1: 0.9693 | Acc: 0.9821

Finales Modell (alle Trainingsdaten)


Training: 100%|██████████| 30/30 [01:33<00:00,  3.11s/epoch, train_f1=0.979, val_f1=0.951, val_acc=0.971]


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/acc,▁▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇████████████
train/f1,▁▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇█████████████
train/loss,█▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
train/lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
val/acc,▁▂▂▂▂▃▄▃▃▅▁▆▅▆▅▇█▆▇▇▇▆▇▇██████
val/f1,▁▂▂▁▃▃▄▄▄▅▃▅▆▆▆▇█▆▇▇▇▆▇▇██████
val/loss,▅▅▄▆▄▄▃▅▂▃█▂▃▂▃▃▂▃▃▂▁▃▃▂▂▂▂▂▂▂
epoch,30
train/acc,0.98647
train/f1,0.97879



ERGEBNIS
  Fold 1: F1=0.9735 | Acc=0.9839
  Fold 2: F1=0.9830 | Acc=0.9899
  Fold 3: F1=0.9765 | Acc=0.9863
  Fold 4: F1=0.9711 | Acc=0.9827
  Fold 5: F1=0.9693 | Acc=0.9821
────────────────────────────────────────
  CV Mean:  F1=0.9747 | Acc=0.9850
  CV Std:   F1=0.0048  | Acc=0.0028
────────────────────────────────────────
  Final:    F1=0.9519 | Acc=0.9713  ← Testset


In [12]:
print("\n" + "="*40)
print("ERGEBNIS")
print("="*40)
for r in fold_results:
    print(f"  Fold {r['fold']}: F1={r['f1']:.4f} | Acc={r['acc']:.4f}")
print(f"{'─'*40}")
print(f"  CV Mean:  F1={np.mean(f1s):.4f} | Acc={np.mean(accs):.4f}")
print(f"  CV Std:   F1={np.std(f1s):.4f}  | Acc={np.std(accs):.4f}")
print(f"{'─'*40}")
print(f"  Final:    F1={final_f1:.4f} | Acc={final_acc:.4f}  (auf Testset)")


ERGEBNIS
  Fold 1: F1=0.9735 | Acc=0.9839
  Fold 2: F1=0.9830 | Acc=0.9899
  Fold 3: F1=0.9765 | Acc=0.9863
  Fold 4: F1=0.9711 | Acc=0.9827
  Fold 5: F1=0.9693 | Acc=0.9821
────────────────────────────────────────
  CV Mean:  F1=0.9747 | Acc=0.9850
  CV Std:   F1=0.0048  | Acc=0.0028
────────────────────────────────────────
  Final:    F1=0.9519 | Acc=0.9713  (auf Testset)


1D-CNN erreicht F1=0.975 (CV) / 0.952 (Testset) bei einer Std von 0.005, daher ist das Training stabil und generalisiert gut. Es dient als starke Baseline für den LSTM-Vergleich. 

# TODO: Confusion matrix